# 11 — Tuning de Hiperparâmetro (CatBoost, XGBoost, LightGBM)
## PRT Seguros

Usamos `RandomizedSearchCV` (busca aleatória, mais eficiente que grid search para muitos
hiperparâmetros) com `scoring="roc_auc"` e 3-fold, sobre o conjunto de features `_v2`
(features com shift removidas + features derivadas do notebook `10`).

**Fluxo por modelo:**
1. Busca aleatória com `n_estimators` fixo e baixo (rápido, só pra explorar a região boa do espaço)
2. Pega os melhores hiperparâmetros
3. Retreina com `n_estimators` alto + early stopping (como nos notebooks `03`-`05`) para achar o número
   ideal de árvores com os melhores hiperparâmetros
4. Avalia no conjunto de validação e gera a submissão Kaggle


## 1. Carregar dados `_v2`

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"
PRT_NAVY, PRT_GREEN, PRT_GRAY = "#19284F", "#39694B", "#737C8A"

train = pd.read_csv("dados_processados/train_model_ready_v2.csv")
val = pd.read_csv("dados_processados/val_model_ready_v2.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready_v2.csv")

feature_cols = [c for c in train.columns if c not in (ID_COL, TARGET_COL)]
X_train, y_train = train[feature_cols], train[TARGET_COL]
X_val, y_val = val[feature_cols], val[TARGET_COL]
X_kaggle = kaggle[feature_cols]

neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Treino: {X_train.shape} | Val: {X_val.shape} | Kaggle: {X_kaggle.shape}")
print(f"scale_pos_weight = {neg_pos_ratio:.2f}")

resultados_tuning = {}


Treino: (80000, 79) | Val: (20000, 79) | Kaggle: (100000, 79)
scale_pos_weight = 7.27


## 2. CatBoost — busca de hiperparâmetros

In [2]:
from catboost import CatBoostClassifier

param_dist_cat = {
    "depth": [4, 5, 6, 7, 8, 9],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "l2_leaf_reg": [1, 3, 5, 7, 9],
    "random_strength": [0.5, 1, 2, 5],
}

busca_cat = RandomizedSearchCV(
    estimator=CatBoostClassifier(
        iterations=300, scale_pos_weight=neg_pos_ratio,
        eval_metric="AUC", random_seed=RANDOM_STATE, verbose=False,
    ),
    param_distributions=param_dist_cat,
    n_iter=15, cv=3, scoring="roc_auc",
    random_state=RANDOM_STATE, n_jobs=3, verbose=1,
)
busca_cat.fit(X_train, y_train)
print(f"Melhores parâmetros CatBoost: {busca_cat.best_params_}")
print(f"Melhor AUC-ROC (CV, 300 iterations): {busca_cat.best_score_:.4f}")


Fitting 3 folds for each of 15 candidates, totalling 45 fits


Melhores parâmetros CatBoost: {'random_strength': 0.5, 'learning_rate': 0.02, 'l2_leaf_reg': 9, 'depth': 6}
Melhor AUC-ROC (CV, 300 iterations): 0.8254


## 3. CatBoost — retreinar com early stopping usando os melhores parâmetros

In [3]:
X_tr_es, X_es, y_tr_es, y_es = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
)
neg_pos_ratio_es = (y_tr_es == 0).sum() / (y_tr_es == 1).sum()

cat_tuned = CatBoostClassifier(
    iterations=3000,
    **busca_cat.best_params_,
    scale_pos_weight=neg_pos_ratio_es,
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    early_stopping_rounds=50,
    verbose=200,
)
cat_tuned.fit(X_tr_es, y_tr_es, eval_set=(X_es, y_es))
print(f"Melhor iteração: {cat_tuned.get_best_iteration()}")

proba_val_cat = cat_tuned.predict_proba(X_val)[:, 1]
auc_cat_tuned = roc_auc_score(y_val, proba_val_cat)
resultados_tuning["catboost_tuned"] = auc_cat_tuned
print(f"AUC-ROC (validação) — CatBoost tuned: {auc_cat_tuned:.4f}")


0:	test: 0.8134715	best: 0.8134715 (0)	total: 25.6ms	remaining: 1m 16s


200:	test: 0.8314228	best: 0.8314228 (200)	total: 5.51s	remaining: 1m 16s


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8323246072
bestIteration = 284

Shrink model to first 285 iterations.
Melhor iteração: 284
AUC-ROC (validação) — CatBoost tuned: 0.8263


## 4. XGBoost — busca de hiperparâmetros

In [4]:
import xgboost as xgb

param_dist_xgb = {
    "max_depth": [3, 4, 5, 6, 7, 8],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5],
}

busca_xgb = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(
        n_estimators=300, scale_pos_weight=neg_pos_ratio,
        tree_method="hist", eval_metric="auc",
        random_state=RANDOM_STATE, n_jobs=1,
    ),
    param_distributions=param_dist_xgb,
    n_iter=15, cv=3, scoring="roc_auc",
    random_state=RANDOM_STATE, n_jobs=3, verbose=1,
)
busca_xgb.fit(X_train, y_train)
print(f"Melhores parâmetros XGBoost: {busca_xgb.best_params_}")
print(f"Melhor AUC-ROC (CV, 300 estimators): {busca_xgb.best_score_:.4f}")


Fitting 3 folds for each of 15 candidates, totalling 45 fits


Melhores parâmetros XGBoost: {'subsample': 0.7, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.02, 'gamma': 0.5, 'colsample_bytree': 0.7}
Melhor AUC-ROC (CV, 300 estimators): 0.8249


## 5. XGBoost — retreinar com early stopping usando os melhores parâmetros

In [5]:
xgb_tuned = xgb.XGBClassifier(
    n_estimators=3000,
    **busca_xgb.best_params_,
    scale_pos_weight=neg_pos_ratio_es,
    tree_method="hist",
    eval_metric="auc",
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_tuned.fit(X_tr_es, y_tr_es, eval_set=[(X_es, y_es)], verbose=200)
print(f"Melhor iteração: {xgb_tuned.best_iteration}")

proba_val_xgb = xgb_tuned.predict_proba(X_val)[:, 1]
auc_xgb_tuned = roc_auc_score(y_val, proba_val_xgb)
resultados_tuning["xgboost_tuned"] = auc_xgb_tuned
print(f"AUC-ROC (validação) — XGBoost tuned: {auc_xgb_tuned:.4f}")


[0]	validation_0-auc:0.80862


[200]	validation_0-auc:0.83097


[400]	validation_0-auc:0.83135


[425]	validation_0-auc:0.83129


Melhor iteração: 375
AUC-ROC (validação) — XGBoost tuned: 0.8253


## 6. LightGBM — busca de hiperparâmetros

In [6]:
import lightgbm as lgb

param_dist_lgb = {
    "num_leaves": [15, 31, 63, 127],
    "max_depth": [-1, 5, 7, 10],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_samples": [10, 20, 30, 50],
}

busca_lgb = RandomizedSearchCV(
    estimator=lgb.LGBMClassifier(
        n_estimators=300, scale_pos_weight=neg_pos_ratio,
        random_state=RANDOM_STATE, n_jobs=1, verbose=-1,
    ),
    param_distributions=param_dist_lgb,
    n_iter=15, cv=3, scoring="roc_auc",
    random_state=RANDOM_STATE, n_jobs=3, verbose=1,
)
busca_lgb.fit(X_train, y_train)
print(f"Melhores parâmetros LightGBM: {busca_lgb.best_params_}")
print(f"Melhor AUC-ROC (CV, 300 estimators): {busca_lgb.best_score_:.4f}")


Fitting 3 folds for each of 15 candidates, totalling 45 fits


Melhores parâmetros LightGBM: {'subsample': 0.6, 'num_leaves': 15, 'min_child_samples': 50, 'max_depth': 7, 'learning_rate': 0.03, 'colsample_bytree': 0.6}
Melhor AUC-ROC (CV, 300 estimators): 0.8250


## 7. LightGBM — retreinar com early stopping usando os melhores parâmetros

In [7]:
lgb_tuned = lgb.LGBMClassifier(
    n_estimators=3000,
    **busca_lgb.best_params_,
    scale_pos_weight=neg_pos_ratio_es,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
lgb_tuned.fit(
    X_tr_es, y_tr_es,
    eval_set=[(X_es, y_es)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)
print(f"Melhor iteração: {lgb_tuned.best_iteration_}")

proba_val_lgb = lgb_tuned.predict_proba(X_val)[:, 1]
auc_lgb_tuned = roc_auc_score(y_val, proba_val_lgb)
resultados_tuning["lightgbm_tuned"] = auc_lgb_tuned
print(f"AUC-ROC (validação) — LightGBM tuned: {auc_lgb_tuned:.4f}")


Melhor iteração: 8
AUC-ROC (validação) — LightGBM tuned: 0.8234


## 8. Comparar com os modelos sem tuning (notebooks `03`-`05`)

In [8]:
resultados_antigos = pd.read_csv("dados_processados/resultados_modelos.csv").set_index("modelo")["auc_roc_val"].to_dict()

print(f"{'Modelo':<24} {'Sem tuning':>12} {'Com tuning':>12} {'Delta':>10}")
for base_name, tuned_name in [("catboost", "catboost_tuned"), ("xgboost", "xgboost_tuned"), ("lightgbm", "lightgbm_tuned")]:
    antigo = resultados_antigos[base_name]
    novo = resultados_tuning[tuned_name]
    print(f"{base_name:<24} {antigo:>12.4f} {novo:>12.4f} {novo - antigo:>+10.4f}")


Modelo                     Sem tuning   Com tuning      Delta
catboost                       0.8254       0.8263    +0.0009
xgboost                        0.8240       0.8253    +0.0013
lightgbm                       0.8238       0.8234    -0.0004


## 9. Gerar submissões dos modelos tunados + registrar resultados

In [9]:
import os
os.makedirs("submissions", exist_ok=True)

modelos_tunados = {
    "catboost_tuned": (cat_tuned, resultados_tuning["catboost_tuned"]),
    "xgboost_tuned": (xgb_tuned, resultados_tuning["xgboost_tuned"]),
    "lightgbm_tuned": (lgb_tuned, resultados_tuning["lightgbm_tuned"]),
}

linhas_resultado = []
for nome, (modelo, auc) in modelos_tunados.items():
    proba_kaggle = modelo.predict_proba(X_kaggle)[:, 1]
    pd.DataFrame({"Id": kaggle[ID_COL], "probabilidade_churn": proba_kaggle}).to_csv(
        f"submissions/submission_{nome}.csv", index=False
    )
    linhas_resultado.append({"modelo": nome, "auc_roc_val": auc})
    print(f"submissions/submission_{nome}.csv salvo (AUC-ROC val = {auc:.4f})")

resultados_path = "dados_processados/resultados_modelos.csv"
resultados = pd.read_csv(resultados_path)
novos = pd.DataFrame(linhas_resultado)
resultados = resultados[~resultados["modelo"].isin(novos["modelo"])]
resultados = pd.concat([resultados, novos], ignore_index=True)
resultados.to_csv(resultados_path, index=False)
print()
print(resultados.sort_values("auc_roc_val", ascending=False).to_string(index=False))


submissions/submission_catboost_tuned.csv salvo (AUC-ROC val = 0.8263)


submissions/submission_xgboost_tuned.csv salvo (AUC-ROC val = 0.8253)


submissions/submission_lightgbm_tuned.csv salvo (AUC-ROC val = 0.8234)

             modelo  auc_roc_val
     catboost_tuned     0.826308
           catboost     0.825413
      xgboost_tuned     0.825254
            xgboost     0.823997
           lightgbm     0.823799
     lightgbm_tuned     0.823361
      random_forest     0.819568
logistic_regression     0.803300
